In [1]:
%load_ext autoreload
%autoreload 2

import gc
import json
import os
import subprocess
from collections import defaultdict

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
import yaml
from dotenv import load_dotenv
from plotly.subplots import make_subplots
from tqdm.auto import tqdm

from notebook_utils.metrics import StepMetricsReader
from runpod_orchestrator import PodOrchestrator, PodOrchestratorConfig
from scaling_llms.data import DataLoaderConfig, get_dataloaders
from scaling_llms.experiments import build_raw_model
from scaling_llms.models import GPTModel
from scaling_llms.registries import (
    DatasetIdentity,
    MakeDatasetRegistryConfig,
    MakeRunRegistryConfig,
    make_dataset_registry,
    make_run_registry,
)
from scaling_llms.utils.io import get_local_repo_dir, load_module_from_path
from scaling_llms.utils.loggers import setup_console_logging
from scaling_llms.utils.training import set_determinism


setup_console_logging()
load_dotenv()

True

In [7]:
# DIRECTORY CONFIGURATION
REPO_ROOT = get_local_repo_dir()
CONFIGS_ROOT = REPO_ROOT / "configs"
EXP_CONFIGS_ROOT = CONFIGS_ROOT / "experiments" /"mup_test"
RUNTIME_CONFIGS_ROOT = CONFIGS_ROOT / "runtime"
LOCAL_ARTIFACTS_ROOT = REPO_ROOT / "tmp" / "artifacts"
R2_PREFIX = "scaling-llms/dev/coord_checks"

# Setup

In [3]:
# EXPERIMENT CONFIGS
configs = load_module_from_path(EXP_CONFIGS_ROOT / "training_curves.py")

In [4]:
# REGISTRIES
reg_configs = yaml.safe_load((RUNTIME_CONFIGS_ROOT / "run_experiments_dev.yaml").read_text())["registries"]
db_url = os.environ.get(reg_configs["database_url_env_name"])
reg_configs["runs"]["artifacts_root"] = str(LOCAL_ARTIFACTS_ROOT / "runs")
reg_configs["datasets"]["artifacts_root"] = str(LOCAL_ARTIFACTS_ROOT / "datasets")

run_registry = make_run_registry(MakeRunRegistryConfig(database_url=db_url, **reg_configs["runs"]))
dataset_registry = make_dataset_registry(MakeDatasetRegistryConfig(database_url=db_url, **reg_configs["datasets"]))

# A. muP Tests

## A.1 Coordinate Check

### Helpers

In [ ]:
def collect_activations(
    model: GPTModel,
    idx: torch.Tensor,
) -> dict[str, torch.Tensor]:
    """
    Run a single forward pass and collect mean absolute activations
    from selected layers for coord-check diagnostics.

    Important:
    - We DO NOT hook `lm_head` because μP scaling is applied AFTER it.
    - Instead, we manually record final logits after `_decode`.
    """

    # Layers we want to probe for activation scaling behavior
    COORD_CHECK_LAYERS = {
        "transformer.wte",        # embedding output
        "transformer.h.0.attn",   # first block attention
        "transformer.h.0.mlp",    # first block MLP
        "transformer.h.0",        # first block residual stream
        "transformer.h.2",        # mid-layer residual stream
        "transformer.h.5",        # late-layer residual stream
        "transformer.norm_f",     # final norm before logits
    }

    acts: dict[str, torch.Tensor] = {}  # stores activations per layer
    hooks = []  # store hook handles to remove later

    # Some modules return tuples → unwrap to tensor
    def _unwrap(out):
        return out[0] if isinstance(out, tuple) else out

    # Register forward hooks on selected modules
    for name, module in model.named_modules():
        if name not in COORD_CHECK_LAYERS:
            continue

        def _hook(m, inp, out, n=name):
            out = _unwrap(out)
            # Store mean absolute activation (simple scalar summary)
            acts[n] = out.detach().abs().mean()

        hooks.append(module.register_forward_hook(_hook))

    # Save current mode and switch to eval for deterministic behavior
    was_training = model.training
    model.eval()

    try:
        with torch.no_grad():
            # Forward pass split into encode/decode
            # so hooks capture internal activations correctly
            x = model._encode(idx)

            # Final logits AFTER μP scaling (critical!)
            logits = model._decode(x)

            # Record final output activations
            acts["logits"] = logits.detach().abs().mean()

    finally:
        # Always clean up hooks to avoid memory leaks
        for h in hooks:
            h.remove()

        # Restore original training mode
        model.train(was_training)

    return acts


def collect_activations_over_batches(
    model: GPTModel,
    batch_iterable,
    num_batches: int,
    device: str | torch.device,
) -> dict[str, torch.Tensor]:
    """
    Estimate expected activations by averaging over multiple batches.

    This reduces noise compared to using a single batch.
    """

    acts_accum = defaultdict(list)  # accumulate activations per layer

    for batch_idx, (x,_) in enumerate(batch_iterable):
        if batch_idx >= num_batches:
            break  # only use a fixed number of batches

        x = x.to(device)

        # Collect activations for this batch
        acts = collect_activations(model, x)

        # Accumulate per-layer activations
        for layer_name, act in acts.items():
            acts_accum[layer_name].append(act)

    # Average across batches → smoother, more reliable estimates
    return {
        layer_name: torch.stack(act_list).mean()
        for layer_name, act_list in acts_accum.items()
    }


def save_to_r2(step2records, save_name):
    serializable = {
        str(step): records
        for step, records in step2records.items()
    }
    remote_path = f"r2:{R2_PREFIX}/{save_name}.json"
    subprocess.run(
        [
            "rclone",
            "rcat",
            remote_path,
            "--s3-no-check-bucket",
        ],
        input=json.dumps(serializable, indent=2).encode(),
        check=True,
    )


def run_coord_check(
    x,
    y,
    eval_dl,
    widths,
    seeds,
    use_mup,
    seq_len,
    vocab_size,
    lr,
    num_steps,
    num_eval_batches,
):
    """
    Main coord-check loop.

    Procedure:
    - Train each model (different widths) on SAME batch
    - Measure activations on MULTIPLE eval batches
    - Aggregate across seeds
    """

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Training batch (fixed across widths for controlled comparison)
    x = x.to(device)
    y = y.to(device)

    step2records = defaultdict(list)

    # Loop over model widths
    for width in tqdm(widths, desc="width"):
        layer_name_step2act_list = defaultdict(list)

        # Loop over random seeds (reduce variance)
        for seed in tqdm(seeds, desc="seed", leave=False):
            set_determinism(seed)

            # Initialize model
            gpt_hparams = dict(
                n_embd=width,
                mup_base_width=configs.BASE_WIDTH if use_mup else None,
                **configs.CONSTANT_GPT_HPARAMS,
            )
            model = build_raw_model(
                seq_len=seq_len,
                vocab_size=vocab_size,
                gpt_hparams=gpt_hparams,
            ).to(device)

            # Create optimizer with μP-aware param groups
            params = model.get_param_groups(base_lr=lr, weight_decay=0.0)
            optimizer = torch.optim.AdamW(params, betas=(0.9, 0.95))

            # Train for a few steps
            for step in tqdm(range(num_steps), desc="step", leave=False):
                model.train()

                # Measure layer activations on MULTIPLE eval batches
                acts = collect_activations_over_batches(
                    model=model,
                    batch_iterable=eval_dl,
                    num_batches=num_eval_batches,
                    device=device,
                )

                # Store activations per layer and step
                for layer_name, act in acts.items():
                    layer_name_step2act_list[(layer_name, step)].append(act)

                if step == num_steps - 1:
                    break  # skip optimization step on last iter

                # Optimize on FIXED training batch
                optimizer.zero_grad()
                out = model(x, y)
                out.loss.backward()
                optimizer.step()

        # Aggregate across seeds -> final statistics
        for (layer_name, step), act_list in layer_name_step2act_list.items():
            step2records[step].append(
                dict(
                    width=width,
                    layer_name=layer_name,
                    mean_abs_act=torch.stack(act_list).mean().item(),
                )
            )
            
    # Save to R2
    save_name = f"coord_check_{'with_mup' if use_mup else 'without_mup'}"
    save_to_r2(step2records, save_name)

    return step2records

### Main execution: run coord-checks with and without muP

In [ ]:
# Fetch dataloaders for coordinate check
dl_config = DataLoaderConfig(
    seq_len=512,
    train_batch_size=16,
    eval_batch_size=16,
    start_sample_idx=0,
    seed=42
)
dls = get_dataloaders(
    dataset_id=DatasetIdentity(**configs.DATASET_KWARGS),
    dataset_registry=dataset_registry,
    dataloader_config=dl_config,
)

# Fix a training batch for the coord check (same across widths)
train_batch = next(iter(dls["train"]))

del dls["train"] 
gc.collect();  # free up memory since we won't use it anymore

In [ ]:
# Run the coordinate check experiment for both standard parameterization and μP
kwargs = dict(
    x=train_batch[0],
    y=train_batch[1],
    eval_dl=dls["eval"],
    widths=configs.WIDTHS,
    seeds=[1, 12, 123, 1234, 12345],
    seq_len=dl_config.seq_len,
    vocab_size=dls["info"]["vocab_size"],
    lr=1e-3,
    num_steps=5,
    num_eval_batches=5,
)

_ = run_coord_check(**kwargs, use_mup=False)
_ = run_coord_check(**kwargs, use_mup=True)

In [ ]:
del dls
gc.collect();  

## A.2 Training Curves


In [ ]:
# Define pod configs and create a pod
pod_cfg = RUNTIME_CONFIGS_ROOT / "pod_orchestrator_training.yaml"
overrides = {
    "pod_spec": {
        "gpu_type_id": "NVIDIA A100-SXM4-80GB", 
        "gpu_count": 2,     
    }
}
orch = PodOrchestrator(PodOrchestratorConfig.from_yaml(pod_cfg, overrides=overrides))
conn = orch.create()

# Run the experiment script on the pod and stream logs back to console
tail_cmd = orch.run(
    script_yaml=RUNTIME_CONFIGS_ROOT / "run_experiments_dev.yaml", # (IMPORTANT: turn on cleanup_after_sync)
    experiment_configs_dir=EXP_CONFIGS_ROOT,
    entry_config_filename="training_curves.py",
    stop_pod_at_success=True,
    stop_pod_at_failure=True,
)
!{tail_cmd} 

# B. Analysis

## B.1 Coord Checks

In [ ]:
def load_from_r2(save_name):
    # Match the exact path structure from save_to_r2
    remote_path = f"r2:{R2_PREFIX}/{save_name}.json"
    
    # Stream the file from R2
    result = subprocess.run(
        [
            "rclone",
            "cat",
            remote_path,
            "--s3-no-check-bucket", # Keeping flags consistent with save_to_r2
        ],
        capture_output=True,
        text=True,
        check=True,
    )
    
    # Parse the captured JSON string
    loaded_data = json.loads(result.stdout)
    
    # Reverse the `str(step)` transformation done during save.
    # This ensures if your steps were integers, they are returned as integers.
    step2records = {
        int(step) if step.isdigit() else step: records
        for step, records in loaded_data.items()
    }
    
    return step2records


def plot_coord_checks(step2records_sp, step2records_mup, steps_to_plot=None):
    steps = steps_to_plot or list(step2records_sp.keys())
    
    # 1. Combine all data into a single DataFrame early
    def build_df(data_dict, param_label):
        dfs = []
        for step in steps:
            df = pd.DataFrame(data_dict.get(step, []))
            if not df.empty:
                df["step"] = step
                df["param"] = param_label
                dfs.append(df)
        return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

    df_all = pd.concat([
        build_df(step2records_sp, "SP"), 
        build_df(step2records_mup, "muP")
    ], ignore_index=True)

    if df_all.empty:
        return go.Figure() # Failsafe for empty data

    # 2. Vectorized cleaning & math (Done ONCE on the whole dataset)
    df_all["layer_name"] = df_all["layer_name"].str.replace("transformer.", "")
    if "log2_width" not in df_all.columns:
        df_all["log2_width"] = np.log2(df_all["width"])

    # 3. Setup Colors and Ticks globally
    all_layers = sorted(df_all["layer_name"].unique())
    colors = px.colors.qualitative.T10
    color_map = {layer: colors[(i + 1) % len(colors)] for i, layer in enumerate(all_layers)}
    
    min_lw = int(df_all["log2_width"].min())
    max_lw = int(df_all["log2_width"].max())
    tick_vals = list(range(min_lw, max_lw + 1))

    # 4. Figure Layout
    fig = make_subplots(
        rows=2, cols=len(steps), 
        shared_xaxes=True, shared_yaxes="rows",
        subplot_titles=[f"Step {s}" for s in steps],
        vertical_spacing=0.08, horizontal_spacing=0.03
    )

    # 5. Plot Traces
    for col_idx, step in enumerate(steps):
        for row_idx, param in enumerate(["SP", "muP"]):
            # Filter the master DataFrame for this specific subplot
            sub_df = df_all[(df_all["step"] == step) & (df_all["param"] == param)]
            
            for layer, grp in sub_df.groupby("layer_name"):
                grp = grp.sort_values("log2_width")
                
                fig.add_trace(
                    go.Scatter(
                        x=grp["log2_width"], y=grp["mean_abs_act"],
                        mode="lines+markers", marker=dict(size=6),
                        line=dict(width=2, color=color_map[layer]),
                        name=layer, legendgroup=layer, 
                        showlegend=(row_idx == 0 and col_idx == 0), # Legend only on top-left
                        opacity=0.7
                    ),
                    row=row_idx + 1, col=col_idx + 1
                )

    # 6. Formatting & Axes
    fig.update_xaxes(tickvals=tick_vals, title_text="log2 width", row=2)

    fig.update_yaxes(title_text="SP<br>mean |act|", row=1, col=1)
    fig.update_yaxes(title_text="muP<br>mean |act|", row=2, col=1)

    fig.update_layout(
        title={
            "text": "SP vs muP: mean |activation| vs width across steps", 
            "y": 0.98, 
            "x": 0.5, 
            "xanchor": "center", 
            "yanchor": "top", 
            "font": dict(size=18)
        },
        legend_title_text="layer", 
        hovermode="x unified", 
        margin=dict(t=100)
    )

    return fig

In [6]:
step2records_sp = load_from_r2("coord_check_without_mup")
step2records_mup = load_from_r2("coord_check_with_mup")

fig = plot_coord_checks(step2records_sp, step2records_mup, steps_to_plot=[0,2,4])
# fig.show()
fig.show(renderer="svg", width=1200, height=600)

NameError: name 'load_from_r2' is not defined

## B.2 Training Curves

In [ ]:
smr = StepMetricsReader(run_registry, experiment_name=configs.EXPERIMENT_NAME)

rows = []
for run_name in smr.list_run_names():
    tmp_df = smr.get_metrics_df(run_name, "train", "nll", "train", "lr")
    tmp_df["width"] = int(run_name.split("_")[1].split("=")[1])
    tmp_df["log2lr"] = tmp_df["lr"].apply(lambda x: np.log2(x))
    tmp_df = tmp_df.groupby(["log2lr", "width"], as_index=False)[["nll"]].min()
    tmp_df["parameterization"] = "muP" if run_name.startswith("mup") else "SP"
    rows.append(tmp_df)

df_min_nll = pd.concat(rows).sort_values(by=["parameterization", "width", "log2lr"])

In [ ]:
# Plot Train Loss vs Log2LR
colors = px.colors.sequential.Reds
widths = sorted(df_min_nll["width"].unique())
width2color = {w: colors[(2*i + 1) % len(colors)] for i, w in enumerate(widths)}

fig = px.line(
    df_min_nll.query("log2lr != -6"),
    x="log2lr",
    y="nll",
    color="width",
    facet_col="parameterization",
    facet_col_spacing=0.05,
    color_discrete_map=width2color,          
    category_orders={
        "width": widths, 
        "parameterization": ["SP", "muP"]
    },
    title="Min Train Loss vs LR",
    labels={"log2lr": "log2 LR"}
)

my_ticks = [-12, -11, -10, -9, -8]
my_tick_strings = [str(t) for t in my_ticks] 

fig.update_xaxes(
    tickvals=my_ticks,
    ticktext=my_tick_strings
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
# fig.show()
fig.show(renderer="svg", width=1200, height=500)

In [ ]:
# Plot Train Loss vs Tokens for select LRs

# Create fig grid
name_log2lr_pairs = [("muP", -3), ("SP", -5), ("SP", -3)]
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[f"{name} LR={2**log2lr:.5f}" for name,log2lr in name_log2lr_pairs]
)
fig.update_yaxes(title_text="Train loss", row=1, col=1)
fig.update_xaxes(title_text="Tokens")

# Add Traces
for col, (name, log2lr) in enumerate(name_log2lr_pairs, start=1):
    tmp_df = name2df[name].query(f"log2lr == {log2lr}")
    for width, g in tmp_df.groupby("width"):
        fig.add_trace(
            go.Scatter(
                x=g["tokens"],
                y=g["train_loss"],
                name=f"width={width}",
                mode="lines",
                legendgroup=f"width={width}",   
                showlegend=(col == 1),         
                line=dict(color=width2color[width])
            ),
            row=1,
            col=col
        )

fig.show()
